# M6.1 — Career Consultant via Structured Output

Solve the vacancy-parsing task from the early modules using LangChain `Structured Output` + a Pydantic schema.

**Input:** `vacancies_messages_50.csv` (columns `text_id`, `text`).

**Output:** `../submissions/m6_1_solution.csv` with columns `text_id, text, job_title, company, salary, tg, grade`.

In [1]:
import os
import warnings
from getpass import getpass

import pandas as pd
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

load_dotenv("../.env")

warnings.filterwarnings('ignore')

In [2]:
if not os.getenv('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass(prompt='Course API key: ')

BASE_URL = 'https://api.vsellm.ru/'
API_KEY = os.getenv('OPENAI_API_KEY')

llm = ChatOpenAI(
    api_key=API_KEY, model='gpt-4o-mini', base_url=BASE_URL, temperature=0.0,
)

In [3]:
df = pd.read_csv('vacancies_messages_50.csv')
print(df.shape)
df.head(2)

(50, 2)


,text_id,text
0,9,#вакансия #vacancy #Python #удаленка #flask #r...
1,31,#ВАКАНСИЯ #Системный_аналитик #РФ\n \nАКЦИОНЕ...


## Pydantic schema

All fields are optional — missing info must come back as `None`. The descriptions encode the formatting rules from the task statement so the model can use them via `with_structured_output`.

In [4]:
class Vacancy(BaseModel):
    """Extract structured information from a single job posting (usually in Russian)."""

    job_title: str | None = Field(
        default=None,
        description=(
            'Job title in the same language as the posting. Strip any grade '
            "words (intern/junior/junior+/middle/middle+/senior/lead and their "
            'Russian equivalents) and the surrounding parentheses/commas. '
            'Examples: "Senior Python developer" -> "Python developer"; '
            '"C++ разработчик (middle, senior)" -> "C++ разработчик". '
            'None if not stated.'
        ),
    )
    company: str | None = Field(
        default=None,
        description=(
            'Company name only, in the same language as the posting. Do NOT '
            'include descriptors like "крупная компания", "финтех", '
            '"мобильная игра". None if not stated.'
        ),
    )
    salary: str | None = Field(
        default=None,
        description=(
            'Salary normalized to a strict format. Rules: '
            '(1) digits without spaces; '
            "(2) convert 'тыс'/'к'/'k' to thousands (multiply by 1000); "
            "(3) drop net/gross/'на руки'/fix/премии/бонусы/% mentions; "
            "(4) range -> '<low>-<high> <cur>' with no spaces around '-'; "
            "(5) only lower bound -> 'от <n> <cur>'; "
            "(6) only upper bound -> 'до <n> <cur>'; "
            "(7) currency suffix is ' руб.' or ' $' (with a single leading space); "
            "(8) hourly rate -> append ' в час'. "
            "Examples: '100к-150к рублей' -> '100000-150000 руб.', "
            "'от 2000$' -> 'от 2000 $', 'до 100к руб' -> 'до 100000 руб.', "
            "'25$/час' -> '25 $ в час'. None if not stated."
        ),
    )
    tg: str | None = Field(
        default=None,
        description=(
            'Telegram contact(s) starting with @, case-sensitive. If multiple '
            "contacts are present, join them with ', ' (comma + single space) "
            'in the order they appear. None if not stated.'
        ),
    )
    grade: str | None = Field(
        default=None,
        description=(
            'One or more grade values from the allowed set: '
            'intern, junior, junior+, middle, middle+, senior, lead. '
            "If several apply, join with ', ' in ascending order "
            '(intern < junior < junior+ < middle < middle+ < senior < lead). '
            'None if not stated.'
        ),
    )

In [5]:
system_prompt = (
    'You extract structured fields from a job posting. Follow the schema field '
    'descriptions strictly. If a field is not clearly stated, return null.\n\n'
    'Critical rules:\n'
    '- job_title must not contain grade words or trailing punctuation left after removal.\n'
    '- company is the bare name, never a descriptor.\n'
    "- salary: digits only (no spaces, no 'тыс'/'к'), 'к' = 1000; range with '-' no spaces; "
    "use 'от <n>' / 'до <n>' for one-sided; currency ' руб.' or ' $'; append ' в час' for hourly. "
    "Never include net/gross/на руки/премия/бонус/%/техника.\n"
    '- tg: case-sensitive @handle, multiple separated by ", ".\n'
    "- grade: subset of {{intern, junior, junior+, middle, middle+, senior, lead}}, ascending, comma+space separated."
)

prompt = ChatPromptTemplate.from_messages([
    ('system', system_prompt),
    ('human', 'Vacancy text:\n\n{text}'),
])

structured_llm = llm.with_structured_output(Vacancy)
chain = prompt | structured_llm

In [6]:
chain.invoke({'text': df.iloc[0]['text']})

Vacancy(job_title='Python developer', company='Collectly', salary='6000-9000 $', tg='@ann_gfio', grade='senior')

In [7]:
records = []
for i, row in df.iterrows():
    try:
        result = chain.invoke({'text': row['text']})
        records.append(result.model_dump())
    except Exception as e:
        print(f'row {i} failed: {e}')
        records.append({k: None for k in Vacancy.model_fields})
    if (i + 1) % 10 == 0:
        print(f'processed {i + 1}/{len(df)}')

parsed = pd.DataFrame(records)
out = pd.concat([df.reset_index(drop=True), parsed], axis=1)
out.head()

processed 10/50
processed 20/50
processed 30/50
processed 40/50
processed 50/50


,text_id,text,job_title,company,salary,tg,grade
0,9,#вакансия #vacancy #Python #удаленка #flask #r...,Python developer,Collectly,6000-9000 $,@ann_gfio,senior
1,31,#ВАКАНСИЯ #Системный_аналитик #РФ\n \nАКЦИОНЕ...,Системный аналитик,ГЛАВНЫЙ НАУЧНЫЙ ИННОВАЦИОННЫЙ ВНЕДРЕНЧЕСКИЙ ЦЕНТР,320000-370000 руб.,@NatalyaMaki,NaN
2,28,#вакансия #vacancy #job #senior #data #DB #dat...,Database Administrator,Match Systems,от 3000 $,@lex_kertis,senior
3,49,#вакансия #fulltime #remote \n\n🔎 Ищем Руковод...,Руководитель отдела системного администрирования,финтех,от 3000 $,@ResearcherRIT,NaN
4,18,#vacancy #job #analyst #travel #sirenatravel #...,Analyst,Sirena Travel,120000-180000 руб.,@ann_gfio,NaN


In [9]:
out_path = '../submissions/m6_1_solution.csv'
out[['text_id', 'text', 'job_title', 'company', 'salary', 'tg', 'grade']].to_csv(
    out_path, index=False,
)
print(f'saved -> {out_path}; rows={len(out)}')

saved -> ../submissions/m6_1_solution.csv; rows=50


In [10]:
out

,text_id,text,job_title,company,salary,tg,grade
0,9,#вакансия #vacancy #Python #удаленка #flask #r...,Python developer,Collectly,6000-9000 $,@ann_gfio,senior
1,31,#ВАКАНСИЯ #Системный_аналитик #РФ\n \nАКЦИОНЕ...,Системный аналитик,ГЛАВНЫЙ НАУЧНЫЙ ИННОВАЦИОННЫЙ ВНЕДРЕНЧЕСКИЙ ЦЕНТР,320000-370000 руб.,@NatalyaMaki,NaN
2,28,#вакансия #vacancy #job #senior #data #DB #dat...,Database Administrator,Match Systems,от 3000 $,@lex_kertis,senior
3,49,#вакансия #fulltime #remote \n\n🔎 Ищем Руковод...,Руководитель отдела системного администрирования,финтех,от 3000 $,@ResearcherRIT,NaN
4,18,#vacancy #job #analyst #travel #sirenatravel #...,Analyst,Sirena Travel,120000-180000 руб.,@ann_gfio,NaN
5,48,#вакансия #fulltime #remote #lookfor #devops\n...,DevOps инженер,Mad Devs,до 5000 $,@recruiter_maddevs,senior
6,41,#вакансия #senior #middle+ #lead #удаленка #оф...,Системный аналитик,Платформа,180000-300000 руб.,@Alexandrabogdanova_96,"middle+, senior, lead"
7,6,#job #python #django #javascript #react #fulls...,Python/Django full-stack разработчик,ivelum,4000-6000 $,@lebedevaoi,NaN
8,17,#вакансия #vacancy #middle #senior #remote #уд...,Go разработчик,КА Алешин Д.А.,до 500000 руб.,@AleshinDmitry80,"middle, senior"
9,4,Вакансия: Разработчик (Vue.js / Golang) для на...,Разработчик (Vue.js / Golang),NaN,NaN,NaN,junior
